### Grades Predictions (for testing accuracy vs existing result)
The code connectes to the server using pyodbc, reads file to make predictions on, retrieves 'pickled' models and makes predictions.  The resulting file is writted out in csv format.

In [ ]:
import pandas as pd
import numpy as np
import pyodbc
from dotenv import load_dotenv
import os

In [2]:
import warnings
warnings.filterwarnings('ignore')

#### Description of the Variables and Structure of the Scoring System
##### Structure of Scoring Equations and Series of Variables comprazing the Scoring Models:

There are three pieces to a catalog scoring equation.<br>
1.	Probability of a Customer Purchasing, Next Year (30%).<br>
2.	Amount a Customer Will Spend if Customer Purchases, Next Year ($100.00).<br>
3.	Percentage of Next Year’s Spend That is Organic (Non-Catalog) (60%).<br>

The catalog score is derived as follows:<br>
•	(Probability of Purchase) * (Predicted Spend) * (1 – Predicted Organic Percentage).<br>


A series of variables comprise the scoring models.<br>
•	ROOT_REC = Square root of months since last purchase.<br>
•	HST_FREQ = Number of life-to-date purchases.<br>
•	HST_DEMD = Life-to-Date Demand spent.<br>
•	HST_AOV = (HST_DEMD / HST_FREQ).<br>
•	HST_AOV000 = HST_AOV / 1000.<br>
•	Merchandise Indicators … 1 = Ever Bought, 0 = Never Bought. Derived from the High Level Category Variable Sent to Me.<br>
o	MR00 = All Other Merchandise<br>
o	MR01 = Adjustments<br>
o	MR02 = Bibles<br>
o	MR03 = Books<br>
o	MR04 = Church Supplies<br>
o	MR05 = Closeouts<br>
o	MR06 = Damaged<br>
o	MR07 = Downloads<br>
o	MR08 = Exclusives<br>
o	MR09 = Gifts<br>
o	MR10 = HomeSchool<br>
o	MR11 = Music<br>
o	MR12 = Videos<br>
•	Channel Indicators … 1 = Ever Bought, 0 = Never Bought. Derived from each attributed channel sent to me.<br>
o	HST_AFFI (affiliates)<br>
o	HST_CATG (catalog)<br>
o	HST_CATI (catalog insert)<br>
o	HST_EMAI (email)<br>
o	HST_SRCO (organic search)<br>
o	HST_OTHR (all other)<br>
o	HST_PBRN (paid search brand)<br>
o	HST_PNON (paid search non-branded)<br>
o	HST_SHOP (shopping / PLA).<br>
o	HST_TEXT (sms)<br>
o	HST_SOCI (social)<br>
•	HST_SHIP (sum all paid shipping, then divide by historical demand spent).<br>
•	HST_ORGN (sum all demand spent except for catalog attribution and catalog insert attribution, divide by sum of all demand spent).<br>
•	HST_CLICK = Sum of all email clicks, past year.<br>
•	HST_VISI = Sum of all website visits, past year.<br><br><br>
•	ROOT_ECL = Square Root of HST_CLCK.   !!!!Edit! Edit!!!!! Edit!!!!! Currently, EOOT_ECL<br>    <br> <br>
•	ROOT_VIS = Square Root of HST_VISI.<br>



#### I. Establish Connection to the Server and read in Input file to make prediction

In [ ]:
# Secure Connection string for Non-trusted connection: reading credentials from credential file
load_dotenv()  # loads .env from current directory or parent directories
mssqlDriver = os.getenv("DB_DRIVER")
mssqlServer = os.getenv("DB_SERVER")
mssqlUser = os.getenv("DB_USERNAME")
mssqlPw = os.getenv("DB_PASSWORD")
mssqlClient = os.getenv("B_DATABASE")
conn_str = f'DRIVER={mssqlDriver};SERVER={mssqlServer};DATABASE={mssqlClient}_CDM;TRUSTED_CONNECTION=no;UID={mssqlUser};PWD={mssqlPw};'

# connect to the SQL Server
conx = pyodbc.connect(conn_str)

In [ ]:
#query = "select * from Model_Data_Prep_20240221"
query = "SELECT t2.*, grade_brand, grade_catalog, grade_catavg, grade_ship,grade_affiliate, grade_email,grade_organicsearch, \
grade_paidbrand, grade_paidnonbrand, grade_shopping, grade_text, grade_social \
from cb_cdr..Score_Output_Dec23 t1 \
left join Model_Data_Prep_20240221 t2 on t2.customerID = t1.household_id "

In [6]:
with pyodbc.connect(conn_str) as conx:                   # no need to close the connection (conx.close())
    cursor = conx.cursor()                               # create cursor
    cursor.execute(query)                                # execute query
    # or, execute query with parameter:
    # cursor.execute(query, 'Active')
    data = cursor.fetchall()                             # fetch the result

In [7]:
pd.set_option('display.max_columns', None)

In [ ]:
# create column list for Model input:
column_list = ['customerID', 'ROOT_REC', 'HST_FREQ', 'HST_DEMD', 'HST_AOV',
       'HST_AOV000', 'MR00', 'MR01', 'MR02', 'MR03', 'MR04', 'MR05', 'MR06',
       'MR07', 'MR08', 'MR09', 'MR10', 'MR11', 'MR12', 'HST_AFFI', 'HST_CATG',
       'HST_CATI', 'HST_EMAI', 'HST_SRCO', 'HST_OTHR', 'HST_PBRN', 'HST_PNON',
       'HST_SHOP', 'HST_TEXT', 'HST_SOCI', 'HST_SHIP', 'HST_ORGN', 'HST_CLICK',
       'HST_VISI', 'ROOT_ECL', 'ROOT_VIS',
        'grade_brand','grade_catalog','grade_catavg','grade_ship', 'grade_affiliate','grade_email','grade_organicsearch',
        'grade_paidbrand','grade_paidnonbrand', 'grade_shopping','grade_text', 'grade_social']
      
    

In [9]:
# define column list for our dataframe, then generate dataframe
#column_list = ['Customer_DIM_ID','Recency_Group','Lifecycle_Group']
df = pd.DataFrame((tuple(t) for t in data), columns = column_list)
df.head()

,customerID,ROOT_REC,HST_FREQ,HST_DEMD,HST_AOV,HST_AOV000,MR00,MR01,MR02,MR03,MR04,MR05,MR06,MR07,MR08,MR09,MR10,MR11,MR12,HST_AFFI,HST_CATG,HST_CATI,HST_EMAI,HST_SRCO,HST_OTHR,HST_PBRN,HST_PNON,HST_SHOP,HST_TEXT,HST_SOCI,HST_SHIP,HST_ORGN,HST_CLICK,HST_VISI,ROOT_ECL,ROOT_VIS,grade_brand,grade_catalog,grade_catavg,grade_ship,grade_affiliate,grade_email,grade_organicsearch,grade_paidbrand,grade_paidnonbrad,grade_shopping,grade_text,grade_social
0,33228738,5.744563,1,12.99,12.990,0.012990,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0.461124,1.000000,0,0,0.000000,0.0,F,F,F,A,F,F,F,F,F,D,D,A
1,33228764,1.414214,2,57.93,28.965,0.028965,0,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0.206801,1.000000,0,0,0.000000,0.0,C,F,D,B,F,F,D,D,D,D,D,F
2,33228780,5.744563,1,7.19,7.190,0.007190,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0.833102,1.000000,0,0,0.000000,0.0,F,F,F,A,F,F,D,F,D,A,F,B
3,33228792,5.744563,1,40.46,40.460,0.040460,0,0,0,1,0,1,0,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0.222195,1.000000,0,0,0.000000,0.0,F,F,F,C,A,D,A,F,D,D,D,F
4,2227982,3.162278,18,1192.14,66.230,0.066230,0,1,1,1,0,1,1,0,0,1,0,0,1,0,1,1,1,0,0,1,1,1,0,0,0.057854,0.941912,2,1,1.414214,1.0,A,D,C,F,D,B,F,B,C,D,B,F


#### II. Retrieve 'pickled' Models, Make Predictions and Calculate Grades
Load saved models; make predictions and calculate customer grades. <br> 
NOTE: Recreate the list of explanatory and response variables if using as separate module

In [11]:
import statsmodels.api as sm

In [12]:
df_prediction = df.copy()
df_prediction = df.fillna(0)

##### 2.1 Define lists of explanatory and response variables for each model; retrieve 'pickled' models and make predictions

In [13]:
# Define lists of expanatory variables for each model
col_predr = ['ROOT_REC','HST_FREQ','HST_AOV000','MR00','MR01','MR03', 'MR04', 'MR05','MR06','MR07','MR08', 'MR09', 'MR10', 'MR11', 'MR12',
            'HST_AFFI', 'HST_CATG', 'HST_CATI', 'HST_EMAI', 'HST_SRCO', 'HST_OTHR','HST_PBRN','HST_PNON', 'HST_SHOP', 'HST_TEXT',
            'ROOT_ECL', 'ROOT_VIS', 'HST_ORGN']
col_preds = ['HST_DEMD', 'ROOT_VIS', 'ROOT_REC', 'ROOT_ECL']
col_predo = ['ROOT_REC','HST_FREQ','HST_ORGN','MR01','MR03','MR07','MR08','MR09', 'MR10','MR11','MR12','HST_AFFI','HST_CATG','HST_EMAI',
            'HST_SRCO','HST_OTHR', 'HST_PBRN', 'HST_PNON', 'HST_SHOP', 'HST_TEXT','HST_SOCI','ROOT_ECL','ROOT_VIS']
col_pred_ship = ['ROOT_REC','HST_FREQ','MR02','MR04','MR05','MR06','MR08','MR10','MR12','HST_AFFI','HST_EMAI','HST_PBRN','HST_PNON',
            'HST_SHOP','HST_TEXT','HST_SOCI','ROOT_ECL','ROOT_VIS','HST_SHIP']
col_pred_affi = ['HST_FREQ','MR01','MR03','MR05','MR06','MR07','MR09','MR10','MR11','HST_AFFI','HST_CATG','HST_CATI','HST_EMAI','HST_OTHR','HST_PBRN',
            'HST_PNON','HST_SHOP','ROOT_ECL','ROOT_VIS','HST_SHIP']

col_pred_emai = ['ROOT_REC','HST_FREQ','MR01','MR02','MR03','MR05','MR06','MR07','MR08','MR09','MR10','MR11','MR12', 'HST_AFFI','HST_CATG',
                 'HST_CATI','HST_EMAI','HST_SRCO','HST_OTHR','HST_PNON','HST_SHOP','HST_TEXT','ROOT_ECL','ROOT_VIS','HST_SHIP']

col_pred_srco = ['ROOT_REC', 'MR01', 'MR03', 'MR04', 'MR07', 'MR08', 'MR09','MR10','HST_AFFI','HST_CATG','HST_CATI','HST_SRCO','HST_PBRN',
            'HST_PNON', 'HST_SHOP','HST_TEXT', 'HST_SOCI', 'ROOT_ECL', 'ROOT_VIS']

col_pred_pbrn = ['ROOT_REC','HST_FREQ','MR01','MR03','MR04', 'MR05', 'MR06', 'MR07','MR08', 'MR10', 'MR11', 'MR12', 'HST_CATG', 'HST_EMAI',
            'HST_SRCO', 'HST_PBRN','HST_PNON', 'HST_SHOP', 'ROOT_ECL', 'ROOT_VIS', 'HST_SHIP']

col_pred_pnon = ['ROOT_REC', 'HST_FREQ', 'MR01','MR03','MR04','MR05', 'MR06', 'MR08', 'MR09', 'MR10', 'HST_AFFI', 'HST_CATG', 'HST_EMAI',
            'HST_SRCO', 'HST_OTHR', 'HST_PBRN','HST_PNON', 'HST_SHOP', 'ROOT_ECL', 'ROOT_VIS', 'HST_SHIP']

col_pred_shop = ['ROOT_REC', 'HST_FREQ', 'MR01','MR02','MR05','MR06', 'MR07', 'MR08', 'MR09', 'MR10', 'MR11', 'HST_AFFI', 'HST_CATG','HST_CATI',
            'HST_EMAI','HST_SRCO','HST_OTHR','HST_PBRN','HST_PNON','HST_SHOP','HST_SOCI','ROOT_ECL','ROOT_VIS','HST_SHIP']

col_pred_text = ['ROOT_REC', 'HST_FREQ', 'MR00','MR01','MR02','MR05', 'MR06', 'MR08', 'MR09', 'MR10', 'HST_AFFI','HST_CATG','HST_EMAI','HST_SRCO','HST_PBRN',
            'HST_SHOP','HST_TEXT','HST_SOCI','ROOT_ECL','ROOT_VIS','HST_SHIP']

col_pred_soci = ['HST_FREQ', 'MR01','MR05','MR07','MR08', 'MR09', 'MR10', 'MR11', 'HST_CATG', 'HST_CATI','HST_SRCO','HST_OTHR','HST_PNON',
            'HST_SOCI','ROOT_ECL','ROOT_VIS','HST_SHIP']

col_list = [col_predr, col_preds, col_predo, col_pred_ship, col_pred_affi, col_pred_emai, col_pred_srco, col_pred_pbrn, col_pred_pnon, col_pred_shop, col_pred_text, col_pred_soci]

In [14]:
# Define list of response variables for each model
prediction_list = ['predr', 'preds', 'predo', 'pred_ship', 'pred_affi','pred_emai', 'pred_srco', 'pred_pbrn', 'pred_pnon', 'pred_shop', 'pred_text', 'pred_soci']

In [15]:
# List with names of 'pickled' models
name_list = ['predr.pickle', 'preds.pickle','predo.pickle', 'pred_ship.pickle', 'pred_affi.pickle', 'pred_emai.pickle','pred_srco.pickle', 
             'pred_pbrn.pickle','pred_pnon.pickle', 'pred_shop.pickle', 'pred_text.pickle', 'pred_soci.pickle']

In [16]:
# Retrieve 'pickled' models:
model_list = []
for name in name_list:
    model = sm.load(name)
    model_list.append(model)
  

In [17]:
# Define function to make predictions
def predictions (df, model_list, col_list, prediction_list):
    for model, col, pred_name in zip(model_list, col_list, prediction_list):
        X_pred = df[col]
        X_pred = sm.add_constant(X_pred)
        y = model.predict(X_pred)
        if pred_name == 'preds':
            df[pred_name] = y
        else:
            df[pred_name] = np.exp(y) / (1 + np.exp(y))

    return df
        

In [18]:
# Call the function:
df_result = predictions(df_prediction, model_list, col_list, prediction_list)

In [19]:
# Add Brand Value (predv) and Catalog Value (predc) – Kevin Attribute Catalog Value = (predk) if needed.   
df_result['predv'] = df_result['predr'] * df_result['preds']
df_result['predc'] = df_result['predv'] * (1 - df_result['predo'])
#df['predk'] = df['predv'] * (1 - df['predx'])


##### 2.2 Use predictions to calculate Grades

In [20]:
df = df_result.copy()

In [21]:
# Compute Grade_Brand
df['Grade_Brand'] = 'F'
df.loc[df['predv'] > 07.5906, 'Grade_Brand'] = 'D'
df.loc[df['predv'] > 15.8716, 'Grade_Brand'] = 'C'
df.loc[df['predv'] > 33.3596, 'Grade_Brand'] = 'B'
df.loc[df['predv'] > 71.8546, 'Grade_Brand'] = 'A'

In [22]:
# Compute Grade_Ship
df['Grade_Ship'] = 'F'
df.loc[df['pred_ship'] > 0.1387, 'Grade_Ship'] = 'D'
df.loc[df['pred_ship'] > 0.1700, 'Grade_Ship'] = 'C'
df.loc[df['pred_ship'] > 0.1949, 'Grade_Ship'] = 'B'
df.loc[df['pred_ship'] > 0.2358, 'Grade_Ship'] = 'A'

In [23]:
# Compute Grade_Affiliate
df['Grade_Affiliate'] = 'F'
df.loc[df['pred_affi'] > 0.0423, 'Grade_Affiliate'] = 'D'
df.loc[df['pred_affi'] > 0.0465, 'Grade_Affiliate'] = 'C'
df.loc[df['pred_affi'] > 0.0533, 'Grade_Affiliate'] = 'B'
df.loc[df['pred_affi'] > 0.1240, 'Grade_Affiliate'] = 'A'

In [24]:
# Compute Grade_Email
df['Grade_Email'] = 'F'
df.loc[df['pred_emai'] > 0.1324, 'Grade_Email'] = 'D'
df.loc[df['pred_emai'] > 0.2031, 'Grade_Email'] = 'C'
df.loc[df['pred_emai'] > 0.3324, 'Grade_Email'] = 'B'
df.loc[df['pred_emai'] > 0.5941, 'Grade_Email'] = 'A'

In [25]:
# Compute Grade_OrganicSearch
df['Grade_OrganicSearch'] = 'F'
df.loc[df['pred_srco'] > 0.1049, 'Grade_OrganicSearch'] = 'D'
df.loc[df['pred_srco'] > 0.1193, 'Grade_OrganicSearch'] = 'C'
df.loc[df['pred_srco'] > 0.2087, 'Grade_OrganicSearch'] = 'B'
df.loc[df['pred_srco'] > 0.2551, 'Grade_OrganicSearch'] = 'A'

In [26]:
# Compute Grade_PaidBrand
df['Grade_PaidBrand'] = 'F'
df.loc[df['pred_pbrn'] > 0.0164, 'Grade_PaidBrand'] = 'D'
df.loc[df['pred_pbrn'] > 0.0316, 'Grade_PaidBrand'] = 'C'
df.loc[df['pred_pbrn'] > 0.0684, 'Grade_PaidBrand'] = 'B'
df.loc[df['pred_pbrn'] > 0.1723, 'Grade_PaidBrand'] = 'A'

In [27]:
# Compute Grade_PaidNonBrand
df['Grade_PaidNonBrand'] = 'F'
df.loc[df['pred_pnon'] > 0.1113, 'Grade_PaidNonBrand'] = 'D'
df.loc[df['pred_pnon'] > 0.1348, 'Grade_PaidNonBrand'] = 'C'
df.loc[df['pred_pnon'] > 0.1622, 'Grade_PaidNonBrand'] = 'B'
df.loc[df['pred_pnon'] > 0.2270, 'Grade_PaidNonBrand'] = 'A'

In [28]:
# Compute Grade_Shopping
df['Grade_Shopping'] = 'F'
df.loc[df['pred_shop'] > 0.2500, 'Grade_Shopping'] = 'D'
df.loc[df['pred_shop'] > 0.3645, 'Grade_Shopping'] = 'C'
df.loc[df['pred_shop'] > 0.4529, 'Grade_Shopping'] = 'B'
df.loc[df['pred_shop'] > 0.5014, 'Grade_Shopping'] = 'A'

In [29]:
# Compute Grade_Text
df['Grade_Text'] = 'F'
df.loc[df['pred_text'] > 0.0291, 'Grade_Text'] = 'D'
df.loc[df['pred_text'] > 0.0351, 'Grade_Text'] = 'C'
df.loc[df['pred_text'] > 0.0445, 'Grade_Text'] = 'B'
df.loc[df['pred_text'] > 0.0646, 'Grade_Text'] = 'A'

In [30]:
# Compute Grade_Social
df['Grade_Social'] = 'F'
df.loc[df['pred_soci'] > 0.0336, 'Grade_Social'] = 'D'
df.loc[df['pred_soci'] > 0.0402, 'Grade_Social'] = 'C'
df.loc[df['pred_soci'] > 0.0521, 'Grade_Social'] = 'B'
df.loc[df['pred_soci'] > 0.2287, 'Grade_Social'] = 'A'

In [31]:
# Compute Grade_Catalog
df['Grade_Catalog'] = 'F'
df.loc[df['predc'] >= 9.0, 'Grade_Catalog'] = 'D'
df.loc[df['predc'] >= 16.0, 'Grade_Catalog'] = 'C'
df.loc[df['predc'] >= 25.0, 'Grade_Catalog'] = 'B'
df.loc[df['predc'] >= 31.0, 'Grade_Catalog'] = 'A'

In [32]:
# Compute Grade_Catavg
df['Grade_Catavg'] = 'F'
df.loc[df['predc'] >= 0.9, 'Grade_Catavg'] = 'D'
df.loc[df['predc'] >= 5.0, 'Grade_Catavg'] = 'C'
df.loc[df['predc'] >= 20.0, 'Grade_Catavg'] = 'B'
df.loc[df['predc'] >= 31.0, 'Grade_Catavg'] = 'A'

In [33]:
# Compute Grade_CatKev
#df['Grade_CatKev'] = 'F'
#df.loc[df['predk'] >= 07.0000, 'Grade_CatKev'] = 'D'
#df.loc[df['predk'] >= 13.9000, 'Grade_CatKev'] = 'C'
#df.loc[df['predk'] >= 25.0000, 'Grade_CatKev'] = 'B'
#df.loc[df['predk'] >= 31.0000, 'Grade_CatKev'] = 'A'

In [36]:
df.head()

,customerID,ROOT_REC,HST_FREQ,HST_DEMD,HST_AOV,HST_AOV000,MR00,MR01,MR02,MR03,MR04,MR05,MR06,MR07,MR08,MR09,MR10,MR11,MR12,HST_AFFI,HST_CATG,HST_CATI,HST_EMAI,HST_SRCO,HST_OTHR,HST_PBRN,HST_PNON,HST_SHOP,HST_TEXT,HST_SOCI,HST_SHIP,HST_ORGN,HST_CLICK,HST_VISI,ROOT_ECL,ROOT_VIS,grade_brand,grade_catalog,grade_catavg,grade_ship,grade_affiliate,grade_email,grade_organicsearch,grade_paidbrand,grade_paidnonbrad,grade_shopping,grade_text,grade_social,predr,preds,predo,pred_ship,pred_affi,pred_emai,pred_srco,pred_pbrn,pred_pnon,pred_shop,pred_text,pred_soci,predv,predc,Grade_Brand,Grade_Ship,Grade_Affiliate,Grade_Email,Grade_OrganicSearch,Grade_PaidBrand,Grade_PaidNonBrand,Grade_Shopping,Grade_Text,Grade_Social,Grade_Catalog,Grade_Catavg
0,33228738,5.744563,1,12.99,12.990,0.012990,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0.461124,1.000000,0,0,0.000000,0.0,F,F,F,A,F,F,F,F,F,D,D,A,0.043893,60.894960,0.920943,0.300995,0.040301,0.110263,0.087877,0.005769,0.092549,0.273965,0.030553,0.296770,2.672850,0.211308,F,A,F,F,F,F,F,D,D,A,F,F
1,33228764,1.414214,2,57.93,28.965,0.028965,0,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0.206801,1.000000,0,0,0.000000,0.0,C,F,D,B,F,F,D,D,D,D,D,F,0.187479,88.713830,0.780901,0.197581,0.035887,0.117670,0.106425,0.026948,0.123195,0.357594,0.029917,0.031022,16.632006,3.644052,C,B,F,F,D,D,D,D,D,F,F,D
2,33228780,5.744563,1,7.19,7.190,0.007190,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0.833102,1.000000,0,0,0.000000,0.0,F,F,F,A,F,F,D,F,D,A,F,B,0.040868,60.570160,0.923807,0.455620,0.037642,0.076662,0.117862,0.004175,0.129156,0.537769,0.016219,0.059003,2.475373,0.188607,F,A,F,F,D,F,D,A,F,B,F,F
3,33228792,5.744563,1,40.46,40.460,0.040460,0,0,0,1,0,1,0,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0.222195,1.000000,0,0,0.000000,0.0,F,F,F,C,A,D,A,F,D,D,D,F,0.057694,62.433280,0.921379,0.177808,0.139208,0.143642,0.260219,0.010881,0.124628,0.309429,0.029851,0.032901,3.602014,0.283195,F,C,A,D,A,F,D,D,D,F,F,F
4,2227982,3.162278,18,1192.14,66.230,0.066230,0,1,1,1,0,1,1,0,0,1,0,0,1,0,1,1,1,0,0,1,1,1,0,0,0.057854,0.941912,2,1,1.414214,1.0,A,D,C,F,D,B,F,B,C,D,B,F,0.611120,152.223011,0.800687,0.131261,0.042009,0.429791,0.074892,0.175425,0.118895,0.231567,0.053624,0.024885,93.026533,18.541426,A,F,F,B,F,A,D,F,B,F,C,C


In [116]:
df.shape

(7881038, 62)

In [117]:
columns = ['customerID','Grade_Brand','Grade_Ship','Grade_Affiliate','Grade_Email','Grade_OrganicSearch',
          'Grade_PaidBrand','Grade_PaidNonBrand','Grade_Shopping','Grade_Text','Grade_Social','Grade_Catalog','Grade_Catavg']
         

In [118]:
#columns = ['customerID','Grade_Brand','Grade_Ship','Grade_Affiliate','Grade_Email','Grade_OrganicSearch',
 #         'Grade_PaidBrand','Grade_PaidNonBrand','Grade_Shopping','Grade_Text','Grade_Social','Grade_Catalog','Grade_Catavg']
#df_grades = df[columns]

In [37]:
#df_grades = df[columns]
df_grades = df.copy()
df_grades.head()

,customerID,ROOT_REC,HST_FREQ,HST_DEMD,HST_AOV,HST_AOV000,MR00,MR01,MR02,MR03,MR04,MR05,MR06,MR07,MR08,MR09,MR10,MR11,MR12,HST_AFFI,HST_CATG,HST_CATI,HST_EMAI,HST_SRCO,HST_OTHR,HST_PBRN,HST_PNON,HST_SHOP,HST_TEXT,HST_SOCI,HST_SHIP,HST_ORGN,HST_CLICK,HST_VISI,ROOT_ECL,ROOT_VIS,grade_brand,grade_catalog,grade_catavg,grade_ship,grade_affiliate,grade_email,grade_organicsearch,grade_paidbrand,grade_paidnonbrad,grade_shopping,grade_text,grade_social,predr,preds,predo,pred_ship,pred_affi,pred_emai,pred_srco,pred_pbrn,pred_pnon,pred_shop,pred_text,pred_soci,predv,predc,Grade_Brand,Grade_Ship,Grade_Affiliate,Grade_Email,Grade_OrganicSearch,Grade_PaidBrand,Grade_PaidNonBrand,Grade_Shopping,Grade_Text,Grade_Social,Grade_Catalog,Grade_Catavg
0,33228738,5.744563,1,12.99,12.990,0.012990,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0.461124,1.000000,0,0,0.000000,0.0,F,F,F,A,F,F,F,F,F,D,D,A,0.043893,60.894960,0.920943,0.300995,0.040301,0.110263,0.087877,0.005769,0.092549,0.273965,0.030553,0.296770,2.672850,0.211308,F,A,F,F,F,F,F,D,D,A,F,F
1,33228764,1.414214,2,57.93,28.965,0.028965,0,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0.206801,1.000000,0,0,0.000000,0.0,C,F,D,B,F,F,D,D,D,D,D,F,0.187479,88.713830,0.780901,0.197581,0.035887,0.117670,0.106425,0.026948,0.123195,0.357594,0.029917,0.031022,16.632006,3.644052,C,B,F,F,D,D,D,D,D,F,F,D
2,33228780,5.744563,1,7.19,7.190,0.007190,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0.833102,1.000000,0,0,0.000000,0.0,F,F,F,A,F,F,D,F,D,A,F,B,0.040868,60.570160,0.923807,0.455620,0.037642,0.076662,0.117862,0.004175,0.129156,0.537769,0.016219,0.059003,2.475373,0.188607,F,A,F,F,D,F,D,A,F,B,F,F
3,33228792,5.744563,1,40.46,40.460,0.040460,0,0,0,1,0,1,0,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0.222195,1.000000,0,0,0.000000,0.0,F,F,F,C,A,D,A,F,D,D,D,F,0.057694,62.433280,0.921379,0.177808,0.139208,0.143642,0.260219,0.010881,0.124628,0.309429,0.029851,0.032901,3.602014,0.283195,F,C,A,D,A,F,D,D,D,F,F,F
4,2227982,3.162278,18,1192.14,66.230,0.066230,0,1,1,1,0,1,1,0,0,1,0,0,1,0,1,1,1,0,0,1,1,1,0,0,0.057854,0.941912,2,1,1.414214,1.0,A,D,C,F,D,B,F,B,C,D,B,F,0.611120,152.223011,0.800687,0.131261,0.042009,0.429791,0.074892,0.175425,0.118895,0.231567,0.053624,0.024885,93.026533,18.541426,A,F,F,B,F,A,D,F,B,F,C,C


In [ ]:
#df_grades.to_csv('Grades_(date).csv')

#### III. Calculate True Positve Rate for Grades and write out files

In [39]:
column_list1 = ['Grade_Brand','Grade_Ship','Grade_Affiliate','Grade_Email','Grade_OrganicSearch',
        'Grade_PaidBrand','Grade_PaidNonBrand','Grade_Shopping','Grade_Text','Grade_Social','Grade_Catalog','Grade_Catavg']

In [42]:
column_list2 = ['grade_brand','grade_ship','grade_affiliate','grade_email','grade_organicsearch',
          'grade_paidbrand','grade_paidnonbrand','grade_shopping','grade_text','grade_social','grade_catalog','grade_catavg']

In [43]:
accuracy_list = []
n = len(column_list1)
for i in range (n):
    accuracy = df_grades[df_grades[column_list1[i]] == df_grades[column_list2[i]]].shape[0] / df_grades.shape[0]
    accuracy_list.append(round(accuracy,2))

In [44]:
accuracy_list

[0.94, 0.93, 0.86, 0.91, 0.94, 0.95, 0.92, 0.92, 0.91, 0.86, 0.96, 0.93]

In [175]:
accuracy_list

[0.94, 0.93, 0.86, 0.92, 0.94, 0.95, 0.91, 0.91, 0.9, 0.87, 0.96, 0.94]

In [176]:
df_accuracy = pd.DataFrame(columns = ['Grade', 'Accuracy Rate'])
df_accuracy['Grade'] = column_list1
df_accuracy['Accuracy Rate'] = accuracy_list

In [177]:
df_accuracy.loc[n,'Grade'] = 'AVERAGE'
df_accuracy.loc[n, 'Accuracy Rate'] = round(np.average(accuracy_list), 2)

In [179]:
df_accuracy

,Grade,Accuracy Rate
0,Grade_Brand,0.94
1,Grade_Ship,0.93
2,Grade_Affiliate,0.86
3,Grade_Email,0.92
4,Grade_OrganicSearch,0.94
5,Grade_PaidBrand,0.95
6,Grade_PaidNonBrand,0.91
7,Grade_Shopping,0.91
8,Grade_Text,0.90
9,Grade_Social,0.87


In [ ]:
#df_grades.to_excel('Grades_100K.xlsx')

In [ ]:
#df_accuracy.to_excel('Accuracy_100K.xlsx')